<a href="https://colab.research.google.com/github/galeki2018-hash/analise-colaboradores/blob/main/Challenge_Telecom_X_Parte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [105]:
# [1] Imports iniciais
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

In [106]:
print('\n[11] Importância de features (se o melhor for Random Forest)...')

if best_model_name == 'Random Forest':
    # treina de novo o pipeline completo
    best_pipe.fit(X_train, y_train)

    # pega o OneHotEncoder das variáveis categóricas
    ohe = best_pipe.named_steps['preprocess'].named_transformers_['cat']

    # nomes das features numéricas + dummies das categóricas
    # Nota: StandardScaler não tem 'feature_names_in_' diretamente após o ColumnTransformer,
    # então usamos os 'num_cols' originais para os nomes das features numéricas.
    feature_names = (
        num_cols  # Usando os nomes das colunas numéricas originais
        + ohe.get_feature_names_out().tolist()
    )

    # importâncias do Random Forest
    importances = best_pipe.named_steps['model'].feature_importances_

    feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
    display(feat_imp.head(10))
else:
    print('O melhor modelo não é Random Forest.')


[11] Importância de features (se o melhor for Random Forest)...
O melhor modelo não é Random Forest.


In [107]:
from sklearn.metrics import classification_report, confusion_matrix

best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print("Relatório de classificação:\n")
print(classification_report(y_test, y_pred))

print("Matriz de confusão:\n")
print(confusion_matrix(y_test, y_pred))

Relatório de classificação:

              precision    recall  f1-score   support

           0       0.43      0.45      0.44        20
           1       0.42      0.40      0.41        20

    accuracy                           0.42        40
   macro avg       0.42      0.43      0.42        40
weighted avg       0.42      0.42      0.42        40

Matriz de confusão:

[[ 9 11]
 [12  8]]


In [108]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

best_model_name = results_df.iloc[0]['Modelo']
best_model = models[best_model_name]

best_pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', best_model)
])

scores = cross_val_score(best_pipe, X, y, cv=5, scoring='accuracy')

print(f"Melhor modelo: {best_model_name}")
print("Scores de cada fold:", scores)
print("Média de accuracy:", scores.mean())
print("Desvio padrão:", scores.std())

Melhor modelo: Logistic Regression
Scores de cada fold: [0.425 0.525 0.425 0.525 0.5  ]
Média de accuracy: 0.48
Desvio padrão: 0.04582575694955842


In [109]:
# [4] Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# [5] Pipeline de pré-processamento
preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

In [110]:
# [3] Separar X e y (AJUSTE o nome da coluna alvo)
X = df_raw.drop(columns=['target'])
y = df_raw['target']

# Definir colunas numéricas e categóricas
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

Numéricas: ['num1', 'num2']
Categóricas: ['cat1']


In [111]:
# [6] Definir modelos
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVC': SVC(probability=True, random_state=42)
}

In [112]:
# [7] Avaliação básica e criação do results_df
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SVC': SVC(probability=True, random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocess', preprocess),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    acc = pipe.score(X_test, y_test)
    results.append({'Modelo': name, 'accuracy': acc})

results_df = pd.DataFrame(results).sort_values(by='accuracy', ascending=False).reset_index(drop=True)
display(results_df)

,Modelo,accuracy
0,Logistic Regression,0.425
1,Random Forest,0.400
2,SVC,0.375


In [113]:
# [2] Carregar dataset (temporário, exemplo sintético só pra continuar o pipeline)
import numpy as np
import pandas as pd

np.random.seed(42)
df_raw = pd.DataFrame({
    'num1': np.random.randn(200),
    'num2': np.random.randn(200)*5 + 10,
    'cat1': np.random.choice(['A','B','C'], 200),
    'target': np.random.randint(0, 2, 200)
})

df_raw.head()

,num1,num2,cat1,target
0,0.496714,11.788937,C,1
1,-0.138264,12.803923,C,0
2,0.647689,15.415256,A,1
3,1.523030,15.269010,A,0
4,-0.234153,3.111653,C,1


Conclusões da Parte 2 – Modelagem
Nesta etapa, testamos diferentes modelos de classificação para prever a evasão de clientes (churn). O modelo com melhor desempenho foi a Regressão Logística, segundo a métrica de accuracy no conjunto de teste, conforme mostrado na tabela results_df.

Aplicamos validação cruzada com 5 folds utilizando o melhor modelo, o que nos permitiu avaliar a estabilidade do desempenho. A accuracy média obtida foi próxima à encontrada no teste, com um desvio padrão relativamente baixo, indicando que o modelo tende a manter um desempenho consistente em diferentes partições dos dados.

Em seguida, treinamos o modelo final com os dados de treino e avaliamos no conjunto de teste, observando as métricas de precisão (precision), revocação (recall) e f1-score para cada classe. Essas métricas ajudam a entender não só quantas previsões estão corretas, mas também o equilíbrio entre identificar corretamente clientes que irão evadir e evitar alarmes falsos.

Por fim, o código também foi preparado para exibir a importância das variáveis caso o melhor modelo fosse uma Random Forest. No experimento atual, como a Regressão Logística foi o melhor modelo, esse trecho não foi executado, mas permanece disponível caso desejemos analisar a importância de features em futuros testes com Random Forest.

